# Recovery Reentry 2021 Trend-Break Cap v1

Notebook-owned diagnostic after `recovery_reentry_cppi_floor_v1` showed the remaining strict-OOS MaxDD is the `2021-04-05 -> 2021-05-12` event, not the later broken-anchor/high-volatility downside regime. This notebook tests whether a high-position trend-break cap can reduce that event without walk-forward overfit.

Governance:
- Baseline: `recovery_reentry_regime_drawdown_guard_v1`
- Direct comparison: `recovery_reentry_cppi_floor_v1`
- Role: `diagnose`
- Changed layer: `execution`
- Target: strict-OOS relative return >= +30pp, strict-OOS MaxDD closer to 15%, no material `wf_2020`, `wf_2022`, or `wf_2023_2024` regression
- Notebook rule: this `.ipynb` is the primary single-run conclusion artifact

External basis:
- Slow Momentum with Fast Reversion / changepoint response: https://arxiv.org/abs/2105.13727
- Deep Momentum Networks: https://arxiv.org/abs/1904.04912
- Volatility Managed Portfolios: https://www.nber.org/papers/w22208
- Taming Momentum Crashes: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2407199


In [1]:
from __future__ import annotations

import itertools
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = Path('/Users/rziiiii/.openclaw/workspace')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ['PYTHONPATH'] = 'src:.'

from src.soxl_quant.app.budget_counterfactual import compute_metrics, dashboard_daily_frame  # noqa: E402
from src.soxl_quant.data import build_dataset, init_qlib  # noqa: E402
from src.soxl_quant.settings import QLIB_DATA_PATH  # noqa: E402

CARD = 'recovery_reentry_2021_trendbreak_cap_v1'
CARD_DIR = ROOT / 'runtime' / 'execution' / 'card' / CARD
CARD_DIR.mkdir(parents=True, exist_ok=True)

BASE_CARD = 'recovery_reentry_regime_drawdown_guard_v1'
CPPI_CARD = 'recovery_reentry_cppi_floor_v1'
BASE_DIR = ROOT / 'runtime' / 'execution' / 'card' / BASE_CARD
CPPI_DIR = ROOT / 'runtime' / 'execution' / 'card' / CPPI_CARD
STRICT_BASE_DAILY = BASE_DIR / f'{BASE_CARD}_soxl_daily.csv'
CPPI_JSON = CPPI_DIR / f'{CPPI_CARD}.json'
BASE_JSON = BASE_DIR / f'{BASE_CARD}.json'
BASE_WF_DAILY = {
    key: BASE_DIR / 'walkforward' / key / f'{BASE_CARD}_{key}_soxl_daily.csv'
    for key in ['wf_2020', 'wf_2022', 'wf_2023_2024']
}

init_qlib(QLIB_DATA_PATH)
dataset = build_dataset()
price = dataset.handler.price_frame.droplevel('instrument')['price'].sort_index()
features = dataset.handler.feature_frame.droplevel('instrument').sort_index()


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))


def read_daily(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    frame['datetime'] = pd.to_datetime(frame['datetime'])
    return frame.sort_values('datetime').reset_index(drop=True)


strict_base = read_daily(STRICT_BASE_DAILY)
wf_base = {key: read_daily(path) for key, path in BASE_WF_DAILY.items()}
base_payload = read_json(BASE_JSON)
cppi_payload = read_json(CPPI_JSON)
base_strict = base_payload['variants'][BASE_CARD]['test_proxy']
cppi_strict = cppi_payload['variants'][CPPI_CARD]['test_proxy']
base_wf = {
    key: read_json(BASE_DIR / 'walkforward' / key / f'{BASE_CARD}_{key}.json')['variants'][f'{BASE_CARD}_{key}']['test_proxy']
    for key in BASE_WF_DAILY
}
print('base_strict', base_strict)
print('cppi_strict', cppi_strict)


[75466:MainThread](2026-05-24 17:11:39,799) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[75466:MainThread](2026-05-24 17:11:40,583) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[75466:MainThread](2026-05-24 17:11:40,583) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[75466:MainThread](2026-05-24 17:11:43,107) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Loading data Done


[75466:MainThread](2026-05-24 17:11:43,109) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[75466:MainThread](2026-05-24 17:11:43,112) INFO - qlib.timer - [log.py:127] - Time cost: 0.006s | Init data Done


base_strict {'rows': 1605, 'start': '2020-01-02', 'end': '2026-05-21', 'return': 2272.853019414519, 'sharpe': 1.2480221658238797, 'max_dd': 32.12017980873844, 'avg_pos': 0.4595798148626328, 'stress_avg_pos': 0.43573123179249773, 'stress_days': 618, 'buy_and_hold_return': 816.5124997770644, 'relative_return': 1456.3405196374547, 'trades': 212}
cppi_strict {'rows': 1605, 'start': '2020-01-02', 'end': '2026-05-21', 'return': 2410.9859762176493, 'sharpe': 1.2858241342240777, 'max_dd': 32.12017980873844, 'avg_pos': 0.4402787696715925, 'stress_avg_pos': 0.3683246346808821, 'stress_days': 662, 'buy_and_hold_return': 816.5124997770644, 'relative_return': 1594.4734764405848, 'trades': 208}


In [2]:
def _price_feature_frame(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame[['datetime', 'position']].copy()
    price_df = price.rename('price').reset_index()
    price_df.columns = ['datetime', 'price']
    out = out.merge(price_df, on='datetime', how='inner')
    feats = features.reset_index().rename(columns={'index': 'datetime'})
    if 'instrument' in feats.columns:
        feats = feats.drop(columns=['instrument'])
    out = out.merge(feats, on='datetime', how='left')
    out = out.sort_values('datetime').reset_index(drop=True)
    out['price_drawdown_21'] = out['price'] / out['price'].rolling(21, min_periods=5).max() - 1.0
    out['price_drawdown_42'] = out['price'] / out['price'].rolling(42, min_periods=10).max() - 1.0
    out['price_drawdown_63'] = out['price'] / out['price'].rolling(63, min_periods=10).max() - 1.0
    out['price_mom_5'] = out['price'] / out['price'].shift(5) - 1.0
    out['price_mom_10'] = out['price'] / out['price'].shift(10) - 1.0
    out['price_mom_21'] = out['price'] / out['price'].shift(21) - 1.0
    out['realized_vol_21_local'] = out['price'].pct_change().rolling(21, min_periods=10).std() * np.sqrt(252)
    return out


def metric_from_position(frame: pd.DataFrame, label: str) -> dict:
    return compute_metrics(frame[['datetime', 'position']], price)


def max_drawdown_event(frame: pd.DataFrame) -> dict:
    scored = frame[['datetime', 'position']].copy()
    scored = scored.merge(price.rename('price').reset_index().rename(columns={'index': 'datetime'}), on='datetime', how='inner')
    positions = scored['position'].to_numpy(dtype=float)
    returns = scored['price'].pct_change().fillna(0.0).to_numpy(dtype=float)
    nav = np.ones(len(scored))
    if len(scored) > 1:
        nav[1:] = np.cumprod(1.0 + positions[:-1] * returns[1:])
    peak = np.maximum.accumulate(nav)
    dd = nav / np.maximum(peak, 1e-12) - 1.0
    trough_idx = int(np.argmin(dd))
    peak_idx = int(np.argmax(nav[: trough_idx + 1]))
    return {
        'peak': scored['datetime'].iloc[peak_idx].date().isoformat(),
        'trough': scored['datetime'].iloc[trough_idx].date().isoformat(),
        'max_dd': abs(float(dd[trough_idx]) * 100.0),
        'peak_position': float(scored['position'].iloc[peak_idx]),
        'trough_position': float(scored['position'].iloc[trough_idx]),
    }


def apply_trendbreak_cap(
    base: pd.DataFrame,
    *,
    drawdown_window: int,
    drawdown_threshold: float,
    max_mom_10: float,
    min_position: float,
    cap_target: float,
    hold_days: int,
    min_realized_vol_21: float | None,
) -> tuple[pd.DataFrame, dict]:
    enriched = _price_feature_frame(base)
    dd_col = f'price_drawdown_{drawdown_window}'
    if dd_col not in enriched.columns:
        raise ValueError(dd_col)
    trigger = (
        (enriched['position'] >= min_position)
        & (enriched[dd_col] <= -abs(drawdown_threshold))
        & (enriched['price_mom_10'] <= max_mom_10)
    )
    if min_realized_vol_21 is not None:
        trigger = trigger & (enriched['realized_vol_21_local'] >= min_realized_vol_21)
    cooldown = 0
    capped_positions = []
    trigger_days = []
    active_days = []
    for is_trigger, base_pos in zip(trigger.fillna(False).to_numpy(bool), enriched['position'].to_numpy(float)):
        if is_trigger:
            cooldown = hold_days
        active = cooldown > 0
        target = min(base_pos, cap_target) if active else base_pos
        capped_positions.append(float(max(target, 0.0)))
        trigger_days.append(bool(is_trigger))
        active_days.append(bool(active))
        if cooldown > 0:
            cooldown -= 1
    out = base.copy()
    out['position'] = capped_positions
    out['trendbreak_cap_trigger'] = trigger_days
    out['trendbreak_cap_active'] = active_days
    out['trendbreak_base_position'] = enriched['position'].to_numpy(float)
    out['trendbreak_price_drawdown'] = enriched[dd_col].to_numpy(float)
    out['trendbreak_price_mom_10'] = enriched['price_mom_10'].to_numpy(float)
    out['trendbreak_realized_vol_21'] = enriched['realized_vol_21_local'].to_numpy(float)
    audit = {
        'trigger_days': int(np.sum(trigger_days)),
        'active_days': int(np.sum(active_days)),
        'changed_days': int(np.sum(np.abs(out['position'].to_numpy(float) - enriched['position'].to_numpy(float)) > 1e-9)),
    }
    return out, audit


def candidate_row(candidate_id: int, params: dict) -> dict:
    capped, audit = apply_trendbreak_cap(strict_base, **params)
    metric = metric_from_position(capped, f'candidate_{candidate_id:03d}')
    event = max_drawdown_event(capped)
    row = {
        'candidate_id': candidate_id,
        **params,
        **metric,
        **audit,
        'maxdd_peak': event['peak'],
        'maxdd_trough': event['trough'],
    }
    row['dd_improvement_vs_base'] = base_strict['max_dd'] - row['max_dd']
    row['return_giveback_vs_base'] = max(base_strict['return'] - row['return'], 0.0)
    row['passes_relative_gate'] = bool(row['relative_return'] >= 30.0)
    row['passes_15dd_gate'] = bool(row['max_dd'] <= 15.0)
    row['strict_score'] = (
        12.0 * max(row['max_dd'] - 15.0, 0.0)
        + 0.02 * row['return_giveback_vs_base']
        + 0.05 * row['changed_days']
        + max(30.0 - row['relative_return'], 0.0) * 5.0
    )
    return row

base_event = max_drawdown_event(strict_base)
print('base_event', base_event)
print(_price_feature_frame(strict_base).query("datetime >= '2021-03-15' and datetime <= '2021-05-20'")[['datetime','position','price_drawdown_21','price_drawdown_42','price_mom_10','mom_10','mom_21','realized_vol_21_local','vix_percentile_252']].tail(15).to_string(index=False))


base_event {'peak': '2021-04-05', 'trough': '2021-05-12', 'max_dd': 32.12017980873848, 'peak_position': 0.7186343814, 'trough_position': 0.8380035218}
  datetime  position  price_drawdown_21  price_drawdown_42  price_mom_10    mom_10    mom_21  realized_vol_21_local  vix_percentile_252
2021-04-30  0.744293          -0.180000          -0.180000     -0.136842 -0.129401 -0.005881               0.858304            0.075397
2021-05-03  0.744293          -0.208667          -0.208667     -0.100758 -0.090676 -0.150759               0.765659            0.075397
2021-05-04  0.744293          -0.241572          -0.245111     -0.099178 -0.089003 -0.257884               0.734643            0.107143
2021-05-05  0.744293          -0.228622          -0.232222     -0.152147 -0.152545 -0.204810               0.736056            0.099206
2021-05-06  0.744293          -0.209645          -0.213333     -0.067930 -0.059967 -0.180208               0.745042            0.079365
2021-05-07  0.838004          -0.

In [3]:
grid = []
for drawdown_window, drawdown_threshold, max_mom_10, min_position, cap_target, hold_days, min_realized_vol_21 in itertools.product(
    [21, 42, 63],
    [0.04, 0.06, 0.08, 0.10],
    [-0.01, -0.03, -0.05, -0.08],
    [0.55, 0.65, 0.75],
    [0.00, 0.10, 0.20, 0.35, 0.50],
    [5, 10, 15, 21],
    [None, 0.75, 1.00],
):
    grid.append({
        'drawdown_window': drawdown_window,
        'drawdown_threshold': drawdown_threshold,
        'max_mom_10': max_mom_10,
        'min_position': min_position,
        'cap_target': cap_target,
        'hold_days': hold_days,
        'min_realized_vol_21': min_realized_vol_21,
    })

rows = [candidate_row(i, params) for i, params in enumerate(grid)]
screen = pd.DataFrame(rows).sort_values(['passes_15dd_gate', 'passes_relative_gate', 'strict_score'], ascending=[False, False, True])
screen_path = CARD_DIR / 'parameter_screen.csv'
screen.to_csv(screen_path, index=False)
eligible = screen[screen['passes_relative_gate'] & (screen['max_dd'] < base_strict['max_dd'])].copy()
top = eligible.sort_values(['passes_15dd_gate', 'strict_score', 'max_dd'], ascending=[False, True, True]).head(16)
if top.empty:
    top = screen.sort_values(['strict_score', 'max_dd']).head(16)
top_path = CARD_DIR / 'parameter_screen_top.csv'
top.to_csv(top_path, index=False)
print(f'parameter_screen rows={len(screen)} target15={int(screen["passes_15dd_gate"].sum())} relative_pass={int(screen["passes_relative_gate"].sum())} improved_dd={len(eligible)}')
print(top[['candidate_id','drawdown_window','drawdown_threshold','max_mom_10','min_position','cap_target','hold_days','min_realized_vol_21','return','relative_return','max_dd','changed_days','strict_score','maxdd_peak','maxdd_trough']].to_string(index=False))


parameter_screen rows=8640 target15=0 relative_pass=3015 improved_dd=1326
 candidate_id  drawdown_window  drawdown_threshold  max_mom_10  min_position  cap_target  hold_days  min_realized_vol_21      return  relative_return    max_dd  changed_days  strict_score maxdd_peak maxdd_trough
          460               21                0.04       -0.05          0.65        0.35         10                 0.75  966.507720       149.995220 30.436657           244    223.566788 2024-03-07   2025-04-08
         1180               21                0.06       -0.05          0.65        0.35         10                 0.75  966.507720       149.995220 30.436657           244    223.566788 2024-03-07   2025-04-08
         1900               21                0.08       -0.05          0.65        0.35         10                 0.75  966.507720       149.995220 30.436657           244    223.566788 2024-03-07   2025-04-08
         3340               42                0.04       -0.05          0.65  

In [4]:
wf_rows = []
for _, candidate in top.iterrows():
    candidate_id = int(candidate['candidate_id'])
    params = {
        'drawdown_window': int(candidate['drawdown_window']),
        'drawdown_threshold': float(candidate['drawdown_threshold']),
        'max_mom_10': float(candidate['max_mom_10']),
        'min_position': float(candidate['min_position']),
        'cap_target': float(candidate['cap_target']),
        'hold_days': int(candidate['hold_days']),
        'min_realized_vol_21': None if pd.isna(candidate['min_realized_vol_21']) else float(candidate['min_realized_vol_21']),
    }
    for wf_key, base_daily in wf_base.items():
        capped, audit = apply_trendbreak_cap(base_daily, **params)
        metric = metric_from_position(capped, f'{CARD}_candidate_{candidate_id:03d}_{wf_key}')
        base = base_wf[wf_key]
        wf_rows.append({
            'candidate_id': candidate_id,
            'wf': wf_key,
            **params,
            **metric,
            **audit,
            'base_return': base['return'],
            'base_max_dd': base['max_dd'],
            'delta_return_vs_base': metric['return'] - base['return'],
            'delta_max_dd_vs_base': metric['max_dd'] - base['max_dd'],
            'wf_return_penalty': max(base['return'] - metric['return'], 0.0),
            'wf_dd_penalty': max(metric['max_dd'] - base['max_dd'], 0.0),
        })

wf_screen = pd.DataFrame(wf_rows)
wf_path = CARD_DIR / 'walkforward_screen.csv'
wf_screen.to_csv(wf_path, index=False)
wf_summary = wf_screen.groupby('candidate_id').agg(
    wf_return_penalty=('wf_return_penalty', 'sum'),
    wf_dd_penalty=('wf_dd_penalty', 'sum'),
    wf_changed_days=('changed_days', 'sum'),
    min_delta_return_vs_base=('delta_return_vs_base', 'min'),
    max_delta_max_dd_vs_base=('delta_max_dd_vs_base', 'max'),
).reset_index()
wf2022 = wf_screen[wf_screen['wf'] == 'wf_2022'][['candidate_id','delta_return_vs_base','delta_max_dd_vs_base']].rename(columns={'delta_return_vs_base':'wf_2022_delta_return','delta_max_dd_vs_base':'wf_2022_delta_max_dd'})
selection = top.merge(wf_summary, on='candidate_id', how='left').merge(wf2022, on='candidate_id', how='left')
selection['wf_2022_ok'] = (selection['wf_2022_delta_return'] >= -3.0) & (selection['wf_2022_delta_max_dd'] <= 1.0)
selection['final_score'] = (
    12.0 * selection['max_dd'].sub(15.0).clip(lower=0.0)
    + 0.02 * selection['return_giveback_vs_base']
    + 0.20 * selection['wf_return_penalty']
    + 3.00 * selection['wf_dd_penalty']
    + 0.03 * selection['changed_days']
    + 0.02 * selection['wf_changed_days']
    + selection['passes_relative_gate'].map({True: 0.0, False: 500.0})
    + selection['wf_2022_ok'].map({True: 0.0, False: 100.0})
)
selection = selection.sort_values(['passes_15dd_gate', 'wf_2022_ok', 'final_score'], ascending=[False, False, True])
selection_path = CARD_DIR / 'selection_table.csv'
selection.to_csv(selection_path, index=False)
selected = selection.iloc[0]
selected_params = {
    'drawdown_window': int(selected['drawdown_window']),
    'drawdown_threshold': float(selected['drawdown_threshold']),
    'max_mom_10': float(selected['max_mom_10']),
    'min_position': float(selected['min_position']),
    'cap_target': float(selected['cap_target']),
    'hold_days': int(selected['hold_days']),
    'min_realized_vol_21': None if pd.isna(selected['min_realized_vol_21']) else float(selected['min_realized_vol_21']),
}
print(f'walkforward_screen rows={len(wf_screen)} wrote={wf_path.relative_to(ROOT)}')
print(selection[['candidate_id','return','relative_return','max_dd','passes_15dd_gate','changed_days','wf_changed_days','wf_2022_ok','wf_return_penalty','wf_dd_penalty','final_score']].to_string(index=False))
print('selected_params', selected_params)


walkforward_screen rows=48 wrote=runtime/execution/card/recovery_reentry_2021_trendbreak_cap_v1/walkforward_screen.csv
 candidate_id      return  relative_return    max_dd  passes_15dd_gate  changed_days  wf_changed_days  wf_2022_ok  wf_return_penalty  wf_dd_penalty  final_score
         1840 1023.710464       207.197964 30.436657             False           268               80       False          49.548537       1.718505   334.927955
         1900  966.507720       149.995220 30.436657             False           244               73       False          48.168947       1.718505   334.936092
         4780  966.507720       149.995220 30.436657             False           244               73       False          48.168947       1.718505   334.936092
         7660  966.507720       149.995220 30.436657             False           244               73       False          48.168947       1.718505   334.936092
         2620  961.998318       145.485818 30.436657             False      

In [5]:
final_daily, final_audit = apply_trendbreak_cap(strict_base, **selected_params)
final_metric = metric_from_position(final_daily, CARD)
final_event = max_drawdown_event(final_daily)
final_payload = {
    'card': CARD,
    'description': 'Notebook-selected high-position trend-break cap focused on the 2021 strict-OOS drawdown event.',
    'baseline_card': BASE_CARD,
    'selected_candidate_id': int(selected['candidate_id']),
    'selected_params': selected_params,
    'baseline_strict': base_strict,
    'selected_strict': final_metric,
    'baseline_drawdown_event': base_event,
    'selected_drawdown_event': final_event,
    'strict_audit': final_audit,
}
json_path = CARD_DIR / f'{CARD}.json'
json_path.write_text(json.dumps(final_payload, indent=2, default=lambda obj: obj.item() if hasattr(obj, 'item') else str(obj)), encoding='utf-8')
daily_path = CARD_DIR / f'{CARD}_soxl_daily.csv'
dashboard_daily_frame(final_daily[['datetime','position']], price).merge(
    final_daily[['datetime','trendbreak_cap_trigger','trendbreak_cap_active','trendbreak_base_position','trendbreak_price_drawdown','trendbreak_price_mom_10']],
    on='datetime',
    how='left',
).to_csv(daily_path, index=False)

wf_artifact_rows = []
for wf_key, base_daily in wf_base.items():
    capped, audit = apply_trendbreak_cap(base_daily, **selected_params)
    metric = metric_from_position(capped, f'{CARD}_{wf_key}')
    out_dir = CARD_DIR / 'walkforward' / wf_key
    out_dir.mkdir(parents=True, exist_ok=True)
    wf_json = out_dir / f'{CARD}_{wf_key}.json'
    wf_daily = out_dir / f'{CARD}_{wf_key}_soxl_daily.csv'
    wf_payload = {
        'card': f'{CARD}_{wf_key}',
        'baseline_card': f'{BASE_CARD}_{wf_key}',
        'selected_params': selected_params,
        'baseline': base_wf[wf_key],
        'selected': metric,
        'audit': audit,
    }
    wf_json.write_text(json.dumps(wf_payload, indent=2, default=lambda obj: obj.item() if hasattr(obj, 'item') else str(obj)), encoding='utf-8')
    dashboard_daily_frame(capped[['datetime','position']], price).merge(
        capped[['datetime','trendbreak_cap_trigger','trendbreak_cap_active','trendbreak_base_position','trendbreak_price_drawdown','trendbreak_price_mom_10']],
        on='datetime',
        how='left',
    ).to_csv(wf_daily, index=False)
    wf_artifact_rows.append({'wf': wf_key, **metric, **audit, 'json_path': str(wf_json.relative_to(ROOT)), 'daily_path': str(wf_daily.relative_to(ROOT))})

wf_artifacts = pd.DataFrame(wf_artifact_rows)
wf_artifacts_path = CARD_DIR / 'walkforward_summary.csv'
wf_artifacts.to_csv(wf_artifacts_path, index=False)
final_payload['selected_wf'] = wf_artifact_rows
json_path.write_text(json.dumps(final_payload, indent=2, default=lambda obj: obj.item() if hasattr(obj, 'item') else str(obj)), encoding='utf-8')
print('final_metric', final_metric)
print('final_event', final_event)
print('final_audit', final_audit)
print('wrote', json_path.relative_to(ROOT), daily_path.relative_to(ROOT), wf_artifacts_path.relative_to(ROOT))


final_metric {'rows': 1605, 'start': '2020-01-02', 'end': '2026-05-21', 'return': 1023.7104639633354, 'sharpe': 1.0803139773595283, 'max_dd': 30.43665681805665, 'avg_pos': 0.40891903850544614, 'stress_avg_pos': 0.3805038971858256, 'stress_days': 672, 'buy_and_hold_return': 816.5124997770644, 'relative_return': 207.19796418627107, 'trades': 207}
final_event {'peak': '2024-03-07', 'trough': '2025-04-08', 'max_dd': 30.43665681805665, 'peak_position': 0.7953301416, 'trough_position': 0.2399999995}
final_audit {'trigger_days': 85, 'active_days': 277, 'changed_days': 268}
wrote runtime/execution/card/recovery_reentry_2021_trendbreak_cap_v1/recovery_reentry_2021_trendbreak_cap_v1.json runtime/execution/card/recovery_reentry_2021_trendbreak_cap_v1/recovery_reentry_2021_trendbreak_cap_v1_soxl_daily.csv runtime/execution/card/recovery_reentry_2021_trendbreak_cap_v1/walkforward_summary.csv


In [6]:
diag_dir = CARD_DIR / 'oos_poor_window_diagnostics'
diag_cmd = [
    sys.executable,
    'scripts/diagnose_oos_poor_windows.py',
    '--daily-csv',
    str(daily_path),
    '--profile',
    CARD,
    '--output-dir',
    str(diag_dir),
    '--oos-start',
    '2020-01-02',
    '--window-size',
    '63',
    '--target-excess',
    '30',
    '--target-max-dd',
    '15',
]
subprocess.run(diag_cmd, cwd=ROOT, check=True)
poor_windows = pd.read_csv(diag_dir / 'poor_windows.csv') if (diag_dir / 'poor_windows.csv').exists() else pd.DataFrame()

wf_lines = []
for row in wf_artifact_rows:
    wf_key = row['wf']
    base = base_wf[wf_key]
    wf_lines.append(
        f"| `{wf_key}` | {base['return']:.2f}% / {base['max_dd']:.2f}% | "
        f"{row['return']:.2f}% / {row['max_dd']:.2f}% | "
        f"{row['return'] - base['return']:.2f}pp / {row['max_dd'] - base['max_dd']:.2f}pp | {int(row['changed_days'])} |"
    )
poor_lines = []
for _, row in poor_windows.head(5).iterrows():
    poor_lines.append(
        f"- {row['start']}..{row['end']}: strategy={row['strategy_return']:.2f}%, "
        f"buyhold={row['buyhold_return']:.2f}%, excess={row['excess_return']:.2f}pp, max_dd={row['max_dd']:.2f}%"
    )
poor_text = '\n'.join(poor_lines) if poor_lines else '- no poor-window rows emitted'

if final_metric['max_dd'] <= 15.0 and final_metric['relative_return'] >= 30.0 and bool(selected['wf_2022_ok']):
    decision = 'Promotion candidate pending broader harness review.'
elif final_metric['max_dd'] < base_strict['max_dd'] and bool(selected['wf_2022_ok']):
    decision = 'Diagnostic keep / no runtime promotion: the trend-break cap reduces strict-OOS drawdown but still misses the 15% target.'
else:
    decision = 'Diagnostic drop / no runtime promotion: strict-OOS or walk-forward evidence is insufficient.'

readout_lines = [
    '## Final Readout',
    '',
    'This notebook is the primary single-run conclusion artifact.',
    '',
    f"- Selected candidate: `{int(selected['candidate_id'])}`",
    f"- Selected params: `{json.dumps(selected_params, sort_keys=True)}`",
    f"- Strict OOS baseline: `{base_strict['return']:.2f}% / {base_strict['sharpe']:.3f} / {base_strict['max_dd']:.2f}% MaxDD / excess {base_strict['relative_return']:.2f}pp`",
    f"- Strict OOS selected: `{final_metric['return']:.2f}% / {final_metric['sharpe']:.3f} / {final_metric['max_dd']:.2f}% MaxDD / excess {final_metric['relative_return']:.2f}pp`",
    f"- Baseline MaxDD event: `{base_event['peak']} -> {base_event['trough']} ({base_event['max_dd']:.2f}%)`",
    f"- Selected MaxDD event: `{final_event['peak']} -> {final_event['trough']} ({final_event['max_dd']:.2f}%)`",
    f"- Trigger/active/changed days: `{final_audit['trigger_days']}` / `{final_audit['active_days']}` / `{final_audit['changed_days']}`",
    f"- Decision: {decision}",
    '',
    '### Walk-Forward vs Regime Baseline',
    '',
    '| window | baseline return / MaxDD | selected return / MaxDD | delta return / delta MaxDD | changed days |',
    '| --- | ---: | ---: | ---: | ---: |',
    *wf_lines,
    '',
    '### Worst Poor Windows',
    '',
    poor_text,
]
display(Markdown('\n'.join(readout_lines)))
print('single_run_conclusion: rendered above in notebook output')


[94387:MainThread](2026-05-24 17:13:51,866) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[94387:MainThread](2026-05-24 17:13:52,596) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.
[94387:MainThread](2026-05-24 17:13:52,596) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[94387:MainThread](2026-05-24 17:13:55,170) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Loading data Done
[94387:MainThread](2026-05-24 17:13:55,170) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done
[94387:MainThread](2026-05-24 17:13:55,170) INFO - qlib.timer - [log.py:127] - Time cost: 0.003s | Init data Done


oos_diagnostic profile=recovery_reentry_2021_trendbreak_cap_v1 windows=12 features=30 oos=2020-01-02..2026-05-21 output=/Users/rziiiii/.openclaw/workspace/runtime/execution/card/recovery_reentry_2021_trendbreak_cap_v1/oos_poor_window_diagnostics


## Final Readout

This notebook is the primary single-run conclusion artifact.

- Selected candidate: `1840`
- Selected params: `{"cap_target": 0.35, "drawdown_threshold": 0.08, "drawdown_window": 21, "hold_days": 10, "max_mom_10": -0.05, "min_position": 0.55, "min_realized_vol_21": 0.75}`
- Strict OOS baseline: `2272.85% / 1.248 / 32.12% MaxDD / excess 1456.34pp`
- Strict OOS selected: `1023.71% / 1.080 / 30.44% MaxDD / excess 207.20pp`
- Baseline MaxDD event: `2021-04-05 -> 2021-05-12 (32.12%)`
- Selected MaxDD event: `2024-03-07 -> 2025-04-08 (30.44%)`
- Trigger/active/changed days: `85` / `277` / `268`
- Decision: Diagnostic drop / no runtime promotion: strict-OOS or walk-forward evidence is insufficient.

### Walk-Forward vs Regime Baseline

| window | baseline return / MaxDD | selected return / MaxDD | delta return / delta MaxDD | changed days |
| --- | ---: | ---: | ---: | ---: |
| `wf_2020` | 113.89% / 26.14% | 83.65% / 26.14% | -30.24pp / 0.00pp | 36 |
| `wf_2022` | -17.13% / 24.22% | -23.98% / 25.94% | -6.85pp / 1.72pp | 8 |
| `wf_2023_2024` | 58.19% / 30.18% | 45.74% / 28.20% | -12.45pp / -1.98pp | 36 |

### Worst Poor Windows

- 2026-02-05..2026-05-06: strategy=24.81%, buyhold=211.46%, excess=-186.65pp, max_dd=4.52%
- 2026-02-10..2026-05-11: strategy=23.44%, buyhold=200.63%, excess=-177.19pp, max_dd=4.52%
- 2026-02-12..2026-05-13: strategy=22.23%, buyhold=191.98%, excess=-169.75pp, max_dd=4.52%
- 2026-02-13..2026-05-14: strategy=21.68%, buyhold=188.26%, excess=-166.58pp, max_dd=4.52%
- 2025-04-08..2025-07-09: strategy=63.77%, buyhold=223.03%, excess=-159.26pp, max_dd=11.50%

single_run_conclusion: rendered above in notebook output
